# 04 · Save/Load & Inference
**Owner:** Member 4

Prove save/load works and build the `predict_risk()` inference function.

In [1]:
# ===== SHARED SETUP (same in every notebook — don't change without telling the team) =====
import os, sys

# In Colab: clone the repo and move into it
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/Ahmed-log11/Deep_Learning_Project.git"
    if not os.path.exists("Deep_Learning_Project"):
        !git clone {REPO_URL}
    %cd Deep_Learning_Project
elif os.path.basename(os.getcwd()) == "dev":
    os.chdir("../..")  # running locally from notebooks/dev

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paths
RAW_CSV          = "data/raw/Car_Insurance_Claim.csv"
PROCESSED_DIR    = "data/processed"
MODELS_DIR       = "models"
FIGURES_DIR      = "reports/figures"
PREPROCESSOR_PKL = f"{MODELS_DIR}/preprocessor.pkl"
BEST_MODEL       = f"{MODELS_DIR}/best_model.keras"
TARGET           = "OUTCOME"

def load_splits():
    """Load the frozen splits saved by notebook 01."""
    names = ["X_train", "X_val", "X_test", "y_train", "y_val", "y_test"]
    return [np.load(f"{PROCESSED_DIR}/{n}.npy") for n in names]


## 1. Load Model & Preprocessor

In [2]:
# ===== Notebook-04 imports =====
import json, tempfile
import joblib
from tensorflow import keras
from sklearn.metrics import roc_auc_score, f1_score

# The final model = the TUNED model from notebook 03 (it had the best validation AUC)
model        = keras.models.load_model(BEST_MODEL, compile=False)   # compile=False: only needed for predicting
preprocessor = joblib.load(PREPROCESSOR_PKL)          # fitted on the training split in notebook 01

# Input rules written by notebook 01 (column names + allowed values)
with open(f"{PROCESSED_DIR}/preprocessor_meta.json") as f:
    META = json.load(f)
INPUT_COLS  = META["raw_input_columns"]
ALLOWED     = META["allowed_values"]
BINARY_COLS = META["column_groups"]["binary"]

print("Model loaded       :", BEST_MODEL, "| expects", model.input_shape[1], "features")
print("Preprocessor loaded:", PREPROCESSOR_PKL, "| takes", len(INPUT_COLS), "raw columns")
print("Raw input columns  :", INPUT_COLS)

Model loaded       : models/best_model.keras | expects 23 features
Preprocessor loaded: models/preprocessor.pkl | takes 17 raw columns
Raw input columns  : ['AGE', 'GENDER', 'RACE', 'DRIVING_EXPERIENCE', 'EDUCATION', 'INCOME', 'CREDIT_SCORE', 'VEHICLE_OWNERSHIP', 'VEHICLE_YEAR', 'MARRIED', 'CHILDREN', 'POSTAL_CODE', 'ANNUAL_MILEAGE', 'VEHICLE_TYPE', 'SPEEDING_VIOLATIONS', 'DUIS', 'PAST_ACCIDENTS']


## 2. Save/Load Check (same predictions before & after)

In [3]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

# 1) The loaded model reproduces the test results that notebook 03 saved
p_loaded = model.predict(X_test, verbose=0).ravel()
with open(f"{MODELS_DIR}/test_evaluation_results.json") as f:
    saved = json.load(f)

auc_now = roc_auc_score(y_test, p_loaded)
f1_now  = f1_score(y_test, (p_loaded >= 0.5).astype(int))
print(f"ROC-AUC   saved in 03: {saved['ROC-AUC']:.4f} | after loading: {auc_now:.4f}")
print(f"F1-Score  saved in 03: {saved['F1-Score']:.4f} | after loading: {f1_now:.4f}")
assert abs(auc_now - saved["ROC-AUC"]) < 1e-3 and abs(f1_now - saved["F1-Score"]) < 1e-3

# 2) Save -> load again: predictions must be exactly the same
tmp_path = os.path.join(tempfile.gettempdir(), "reload_check.keras")
model.save(tmp_path)
reloaded   = keras.models.load_model(tmp_path, compile=False)
p_reloaded = reloaded.predict(X_test, verbose=0).ravel()
print("\nMax difference in predictions before vs after save/load:", np.abs(p_loaded - p_reloaded).max())
assert np.allclose(p_loaded, p_reloaded, atol=1e-6)

print("\nSave/load check passed.")

ROC-AUC   saved in 03: 0.9027 | after loading: 0.9027
F1-Score  saved in 03: 0.7430 | after loading: 0.7430



Max difference in predictions before vs after save/load: 0.0

Save/load check passed.


## 3. `predict_risk(input_dict)` Function

In [4]:
THRESHOLD = 0.5   # same threshold used for the test evaluation in notebook 03

def predict_risk(input_dict, threshold=THRESHOLD):
    """Predict the claim probability for ONE customer.

    input_dict : raw values for the 17 columns in INPUT_COLS,
                 e.g. {"AGE": "16-25", "GENDER": "male", ..., "PAST_ACCIDENTS": 0}
    returns    : {"claim_probability": float, "prediction": int}   (1 = likely to claim)
    """
    # 1) Check the input (the preprocessor would silently accept wrong categories)
    missing = [c for c in INPUT_COLS if c not in input_dict]
    extra   = [c for c in input_dict if c not in INPUT_COLS]
    if missing or extra:
        raise ValueError(f"Missing columns: {missing} | Unknown columns: {extra}")

    row = {}
    for c in INPUT_COLS:
        v = input_dict[c]
        if isinstance(v, str):
            v = v.strip().lower()                    # same text cleaning as notebook 01
        if c in ALLOWED and v not in ALLOWED[c]:
            raise ValueError(f"{c}={v!r} is not allowed. Allowed values: {ALLOWED[c]}")
        if c in BINARY_COLS and v not in (0, 1):
            raise ValueError(f"{c} must be 0 or 1, got {v!r}")
        row[c] = v

    # 2) Same preprocessing as training  ->  3) model
    X = preprocessor.transform(pd.DataFrame([row], columns=INPUT_COLS)).astype("float32")
    prob = float(model.predict(X, verbose=0)[0, 0])
    return {"claim_probability": round(prob, 4), "prediction": int(prob >= threshold)}

## 4. Test on Sample Customers

In [5]:
# 1) Real customers from the test set: predict_risk() must give the same result as the model on X_test
raw = pd.read_csv(f"{PROCESSED_DIR}/splits_raw.csv")
test_raw = raw[raw["split"] == "test"].reset_index(drop=True)     # same row order as X_test

# 3 customers who did NOT claim + 3 who DID claim
sample_ids = list(test_raw.index[test_raw[TARGET] == 0][:3]) + list(test_raw.index[test_raw[TARGET] == 1][:3])

rows = []
for i in sample_ids:
    customer = test_raw.loc[i, INPUT_COLS].to_dict()
    result = predict_risk(customer)
    rows.append({"customer": i, "actual": int(test_raw.loc[i, TARGET]), **result,
                 "model_on_X_test": round(float(p_loaded[i]), 4)})
check = pd.DataFrame(rows)
display(check)
assert np.allclose(check["claim_probability"], check["model_on_X_test"], atol=1e-3)
print("predict_risk() matches the model on X_test for these customers.")

,customer,actual,claim_probability,prediction,model_on_X_test
0,0,0,0.0059,0,0.0059
1,1,0,0.0711,0,0.0711
2,2,0,0.0332,0,0.0332
3,10,1,0.3190,0,0.3190
4,13,1,0.6117,1,0.6117
5,18,1,0.4531,0,0.4531


predict_risk() matches the model on X_test for these customers.


In [6]:
# 2) Two made-up customers
low_risk = {
    "AGE": "40-64", "GENDER": "female", "RACE": "majority", "DRIVING_EXPERIENCE": "20-29y",
    "EDUCATION": "university", "INCOME": "upper class", "CREDIT_SCORE": 0.75,
    "VEHICLE_OWNERSHIP": 1, "VEHICLE_YEAR": "after 2015", "MARRIED": 1, "CHILDREN": 1,
    "POSTAL_CODE": 10238, "ANNUAL_MILEAGE": 10000, "VEHICLE_TYPE": "sedan",
    "SPEEDING_VIOLATIONS": 0, "DUIS": 0, "PAST_ACCIDENTS": 0,
}
# Same customer, but young, new driver, low income, doesn't own the car, older vehicle, more mileage
high_risk = {**low_risk, "AGE": "16-25", "DRIVING_EXPERIENCE": "0-9y", "INCOME": "poverty",
             "CREDIT_SCORE": 0.30, "VEHICLE_OWNERSHIP": 0, "VEHICLE_YEAR": "before 2015",
             "MARRIED": 0, "CHILDREN": 0, "ANNUAL_MILEAGE": 18000}

for name, customer in [("Low-risk customer ", low_risk), ("High-risk customer", high_risk)]:
    print(name, "->", predict_risk(customer))

Low-risk customer  -> {'claim_probability': 0.0094, 'prediction': 0}
High-risk customer -> {'claim_probability': 0.9053, 'prediction': 1}


In [7]:
# 3) Wrong input is rejected with a clear message
try:
    predict_risk({**low_risk, "AGE": "30"})
except ValueError as e:
    print("Rejected:", e)

Rejected: AGE='30' is not allowed. Allowed values: ['16-25', '26-39', '40-64', '65+']


### Summary
- `best_model.keras` (tuned model from notebook 03) and `preprocessor.pkl` (notebook 01) load correctly and reproduce the saved test results.
- Saving and loading again gives identical predictions.
- `predict_risk()` takes one customer's raw values, checks them, applies the **same** preprocessing as training, and returns the claim probability + 0/1 prediction (threshold 0.5).